# sample8 - LangGraph 基礎

**LangGraph** は、状態（State）を共有する処理（Node）と接続（Edge）でワークフローを作るライブラリです。
sample7 のエージェントでは LLM がツールを選びました。ここでは自分で処理の順序や分岐条件を定義します。

| ステップ | 内容 |
|----------|------|
| 0 | ライブラリの確認 |
| 1 | State・Node・Edge の基本 |
| 2 | 条件分岐 |
| 3 | 繰り返しと終了条件 |
| 4 | 状態の保存 |
| 5 | Ollama をノードに組み込む |

ステップ 0〜4 は LLM・Docker・モデルのダウンロードなしで実行できます。
ステップ 5 では sample6 と同じ Ollama の `llama3` を使います。

## 事前準備

Python 3.10 以上、LangGraph 1.x を使います。VS Code で `env` のカーネルを選びます。

```bash
source env/bin/activate  # リポジトリのルートで実行
python -m pip install "langgraph>=1,<2"
```

ステップ 5 も実行する場合は、次の準備が必要です。

```bash
python -m pip install "langchain-ollama>=1,<2"
docker start ollama  # 停止中の場合
docker exec -it ollama ollama pull llama3
```

ノートブック内でインストールする場合は `%pip install` を使い、完了後にカーネルを再起動してください。

## 0. ライブラリの確認

In [ ]:
from importlib.metadata import version
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

print("LangGraph バージョン:", version("langgraph"))

## 1. State・Node・Edge の基本

State はノード間で渡すデータです。`TypedDict` でキーと値の型を示します。
各ノードは状態を受け取り、更新する項目を辞書で返します。状態全体を書き換える必要はありません。
通常の項目は返された値で上書きされます。

ここでは `START → normalize → greet → END` の順に処理します。
`compile()` で実行可能なグラフを作り、`invoke()` で実行します。

In [ ]:
class GreetingState(TypedDict):
    name: str
    greeting: str


def normalize_name(state: GreetingState) -> dict:
    return {"name": state["name"].strip() or "ゲスト"}


def make_greeting(state: GreetingState) -> dict:
    return {"greeting": f"こんにちは、{state['name']}さん。"}


builder = StateGraph(GreetingState)
builder.add_node("normalize", normalize_name)
builder.add_node("greet", make_greeting)
builder.add_edge(START, "normalize")
builder.add_edge("normalize", "greet")
builder.add_edge("greet", END)
greeting_graph = builder.compile()

result = greeting_graph.invoke({"name": "  太郎  ", "greeting": ""})
print(result)

## 2. 条件分岐

`add_conditional_edges` に、次に進む先を選ぶ関数を渡します。
この例は入力中の「計算」という文字で分岐する固定ルールです。LLM による意図の判定ではありません。
両方の入力を実行して経路の違いを確認します。

In [ ]:
class RouteState(TypedDict):
    question: str
    route: str
    answer: str


def classify(state: RouteState) -> dict:
    return {"route": "calculation" if "計算" in state["question"] else "guide"}


def choose_route(state: RouteState) -> Literal["calculation", "guide"]:
    return "calculation" if state["route"] == "calculation" else "guide"


def calculation_help(state: RouteState) -> dict:
    return {"answer": "計算には sample7 の multiply ツールを使えます。"}


def general_help(state: RouteState) -> dict:
    return {"answer": "LangGraph ではノードとエッジで処理の流れを定義します。"}


route_builder = StateGraph(RouteState)
route_builder.add_node("classify", classify)
route_builder.add_node("calculation", calculation_help)
route_builder.add_node("guide", general_help)
route_builder.add_edge(START, "classify")
route_builder.add_conditional_edges(
    "classify", choose_route, {"calculation": "calculation", "guide": "guide"},
)
route_builder.add_edge("calculation", END)
route_builder.add_edge("guide", END)
route_graph = route_builder.compile()

for question in ["掛け算の計算をしたい", "LangGraph を教えて"]:
    result = route_graph.invoke({"question": question, "route": "", "answer": ""})
    print("質問:", question)
    print("分岐:", result["route"], "回答:", result["answer"])

## 3. 繰り返しと終了条件

エッジを前のノードへ戻すと繰り返しになります。必ず終了条件を用意します。
この例ではカウントが目標に達するまで 1 ずつ増やします。
最初に判定するので、開始時点で目標に達していれば加算しません。
`stream_mode="updates"` では、各ノードが返した更新を順番に確認できます。

In [ ]:
class CounterState(TypedDict):
    count: int
    target: int


def increment(state: CounterState) -> dict:
    return {"count": state["count"] + 1}


def continue_counting(state: CounterState) -> Literal["increment", "done"]:
    return "increment" if state["count"] < state["target"] else "done"


counter_builder = StateGraph(CounterState)
counter_builder.add_node("increment", increment)
paths = {"increment": "increment", "done": END}
counter_builder.add_conditional_edges(START, continue_counting, paths)
counter_builder.add_conditional_edges("increment", continue_counting, paths)
counter_graph = counter_builder.compile()

for update in counter_graph.stream(
    {"count": 0, "target": 3},
    config={"recursion_limit": 10},
    stream_mode="updates",
):
    print(update)

print("最初から目標に達している場合:", counter_graph.invoke({"count": 3, "target": 3}))

## 4. 状態の保存

`compile(checkpointer=...)` で保存先を設定し、`thread_id` で状態を区別します。
保存したグラフに同じ ID で新しい入力を渡すと、前回の状態を引き継いで START から実行します。
ここでは `count` を引き継ぎ、`target` だけを更新します。

`InMemorySaver` はカーネル内のメモリに保存します。再起動すると消えます。
このセルの再実行でも保存先を作り直します。

In [ ]:
memory = InMemorySaver()
persistent_counter = counter_builder.compile(checkpointer=memory)
config = {"configurable": {"thread_id": "sample8-counter"}, "recursion_limit": 10}

first = persistent_counter.invoke({"count": 0, "target": 2}, config=config)
print("1回目:", first)
print("保存状態:", persistent_counter.get_state(config).values)

second = persistent_counter.invoke({"target": 4}, config=config)
print("2回目（count を継承）:", second)

other_config = {"configurable": {"thread_id": "another-counter"}, "recursion_limit": 10}
print("別の ID:", persistent_counter.invoke({"count": 0, "target": 1}, config=other_config))
print("元の ID の状態:", persistent_counter.get_state(config).values)

## 5. Ollama をノードに組み込む

ここからは Ollama への接続が必要です。
sample6 の `プロンプト | LLM | 出力パーサー` を 1 つのノードに組み込みます。
「説明を生成 → 説明を要約」の 2 段階で、それぞれの結果を状態に保存します。
このグラフは LLM を 2 回呼び出します。ツール呼び出しを使わないため、`llama3` を利用できます。

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="llama3", base_url="http://localhost:11434", temperature=0)
explain_chain = (
    ChatPromptTemplate.from_template("{topic} を初心者向けに日本語で3文で説明してください。")
    | llm
    | StrOutputParser()
)
summary_chain = (
    ChatPromptTemplate.from_template("次の説明を日本語の1文に要約してください。\n{explanation}")
    | llm
    | StrOutputParser()
)


class LessonState(TypedDict):
    topic: str
    explanation: str
    summary: str


def explain(state: LessonState) -> dict:
    return {"explanation": explain_chain.invoke({"topic": state["topic"]})}


def summarize(state: LessonState) -> dict:
    return {"summary": summary_chain.invoke({"explanation": state["explanation"]})}


lesson_builder = StateGraph(LessonState)
lesson_builder.add_node("explain", explain)
lesson_builder.add_node("summarize", summarize)
lesson_builder.add_edge(START, "explain")
lesson_builder.add_edge("explain", "summarize")
lesson_builder.add_edge("summarize", END)
lesson_graph = lesson_builder.compile()

lesson = lesson_graph.invoke({"topic": "RAG", "explanation": "", "summary": ""})
print("説明:", lesson["explanation"])
print("要約:", lesson["summary"])

## 学習した構成の使い分け

| サンプル | 構成 | 処理の流れ |
|----------|------|------------|
| sample6 | LangChain のチェーン | 決められた順序で処理する |
| sample7 | LangChain のエージェント | LLM がツールを選び、結果を使って回答する |
| sample8 | LangGraph | 状態・分岐・繰り返しをコードで定義する |

LangChain の `create_agent` も内部では LangGraph を利用します。
LangGraph のノードには、通常の Python 関数、チェーン、エージェントなどを組み込めます。

## 参考資料

- [LangGraph Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api)
- [LangGraph の状態保存](https://docs.langchain.com/oss/python/langgraph/persistence)
- [ChatOllama](https://docs.langchain.com/oss/python/integrations/chat/ollama)